# Silver Layer: Taxi Zones Validation

Validates row preservation, lineage completeness, LocationID uniqueness and range, blank descriptions, and Green Taxi referential integrity.

**Prerequisite:** Run `01_silver_green_taxi.ipynb` and `02_silver_taxi_zones.ipynb` first.


## 2. Taxi Zones Validation

Validation checks for the Silver `taxi_zones` table:
- Source vs Silver row count
- NULL checks on business columns
- Duplicate LocationID check
- LocationID range and distinct-count check
- Blank/space-only Zone profiling
- Pickup/dropoff referential integrity checks against Silver Green Taxi
- Final PASS/FAIL validation summary

In [0]:
%sql
-- Compare source and Silver row counts
SELECT
    'Source (Bronze)' AS table_name,
    COUNT(*) AS row_count
FROM `ftw-week-08`.`01_bronze`.`taxi_zones`

UNION ALL

SELECT
    'Target (Silver)' AS table_name,
    COUNT(*) AS row_count
FROM `ftw-week-08`.`02_silver`.`taxi_zones`

In [0]:
%sql
-- Profile special categorical values in Taxi Zones

SELECT
    LocationID,
    Borough,
    Zone,
    service_zone
FROM `ftw-week-08`.`02_silver`.`taxi_zones`
WHERE UPPER(TRIM(COALESCE(Borough, ''))) IN ('N/A', 'UNKNOWN')
   OR UPPER(TRIM(COALESCE(Zone, ''))) IN ('N/A', 'OUTSIDE OF NYC')
   OR UPPER(TRIM(COALESCE(service_zone, ''))) = 'N/A'
ORDER BY LocationID;

FROM `ftw-week-08`.`02_silver`.`taxi_zones`;

In [0]:
-- Identify rows containing N/A values

SELECT
    LocationID,
    Borough,
    Zone,
    service_zone,
    source_system,
    source_file,
    batch_id,
    ingested_at
FROM `ftw-week-08`.`02_silver`.`taxi_zones`
WHERE UPPER(TRIM(COALESCE(Borough, ''))) = 'N/A'
   OR UPPER(TRIM(COALESCE(Zone, ''))) = 'N/A'
   OR UPPER(TRIM(COALESCE(service_zone, ''))) = 'N/A'
ORDER BY LocationID;

In [0]:
%sql
-- Profile categorical values in Silver Taxi Zones

SELECT
    'Borough' AS column_name,
    Borough AS value,
    COUNT(*) AS row_count
FROM `ftw-week-08`.`02_silver`.`taxi_zones`
GROUP BY Borough

UNION ALL

SELECT
    'Zone' AS column_name,
    Zone AS value,
    COUNT(*) AS row_count
FROM `ftw-week-08`.`02_silver`.`taxi_zones`
GROUP BY Zone

UNION ALL

SELECT
    'service_zone' AS column_name,
    service_zone AS value,
    COUNT(*) AS row_count
FROM `ftw-week-08`.`02_silver`.`taxi_zones`
GROUP BY service_zone

ORDER BY column_name, value;

#### Service Zone Domain Check: PASS 
All 265 Silver Taxi Zone records contain service_zone values within the expected source-defined domain. No unexpected categorical values were found.

In [0]:
-- Validate service_zone against expected source-defined values

SELECT
    service_zone,
    COUNT(*) AS row_count
FROM `ftw-week-08`.`02_silver`.`taxi_zones`
WHERE service_zone NOT IN (
    'EWR',
    'Boro Zone',
    'Yellow Zone',
    'Airports',
    'N/A'
)
GROUP BY service_zone;

In [0]:
%sql
-- Check for duplicate LocationIDs
SELECT
    LocationID,
    COUNT(*) AS record_count
FROM `ftw-week-08`.`02_silver`.`taxi_zones`
GROUP BY LocationID
HAVING COUNT(*) > 1

In [0]:
%sql
-- Check LocationID range and distinct count
SELECT
    MIN(LocationID) AS min_location_id,
    MAX(LocationID) AS max_location_id,
    COUNT(DISTINCT LocationID) AS distinct_location_ids
FROM `ftw-week-08`.`02_silver`.`taxi_zones`

In [0]:
%sql
-- Validate Silver Green Taxi references against Silver Taxi Zones

SELECT
    'pickup' AS location_type,
    COUNT(*) AS orphan_rows
FROM `ftw-week-08`.`02_silver`.`green_taxi` g
LEFT JOIN `ftw-week-08`.`02_silver`.`taxi_zones` z
    ON g.PULocationID = z.LocationID
WHERE g.PULocationID IS NOT NULL
  AND z.LocationID IS NULL

UNION ALL

SELECT
    'dropoff' AS location_type,
    COUNT(*) AS orphan_rows
FROM `ftw-week-08`.`02_silver`.`green_taxi` g
LEFT JOIN `ftw-week-08`.`02_silver`.`taxi_zones` z
    ON g.DOLocationID = z.LocationID
WHERE g.DOLocationID IS NOT NULL
  AND z.LocationID IS NULL;

In [0]:
%sql
-- Final validation summary with Pass/Fail status
WITH validation_results AS (
    SELECT
        -- Row counts
        (SELECT COUNT(*)
         FROM `ftw-week-08`.`01_bronze`.`taxi_zones`) AS source_count,

        (SELECT COUNT(*)
         FROM `ftw-week-08`.`02_silver`.`taxi_zones`) AS silver_count,

        -- NULL checks
        (SELECT COUNT_IF(LocationID IS NULL)
         FROM `ftw-week-08`.`02_silver`.`taxi_zones`) AS null_location_id,

        (SELECT COUNT_IF(Borough IS NULL)
         FROM `ftw-week-08`.`02_silver`.`taxi_zones`) AS null_borough,

        (SELECT COUNT_IF(Zone IS NULL)
         FROM `ftw-week-08`.`02_silver`.`taxi_zones`) AS null_zone,

        (SELECT COUNT_IF(service_zone IS NULL)
         FROM `ftw-week-08`.`02_silver`.`taxi_zones`) AS null_service_zone,

        -- Lineage metadata NULL checks
        (SELECT COUNT_IF(source_system IS NULL)
         FROM `ftw-week-08`.`02_silver`.`taxi_zones`) AS null_source_system,

        (SELECT COUNT_IF(source_file IS NULL)
         FROM `ftw-week-08`.`02_silver`.`taxi_zones`) AS null_source_file,

        (SELECT COUNT_IF(batch_id IS NULL)
         FROM `ftw-week-08`.`02_silver`.`taxi_zones`) AS null_batch_id,

        (SELECT COUNT_IF(ingested_at IS NULL)
         FROM `ftw-week-08`.`02_silver`.`taxi_zones`) AS null_ingested_at,

        -- Categorical domain validation
        (SELECT COUNT(*)
         FROM `ftw-week-08`.`02_silver`.`taxi_zones`
         WHERE service_zone NOT IN (
             'EWR',
             'Boro Zone',
             'Yellow Zone',
             'Airports',
             'N/A'
         )) AS invalid_service_zone,

        -- Uniqueness
        (SELECT COUNT(*)
         FROM (
             SELECT LocationID, COUNT(*) AS cnt
             FROM `ftw-week-08`.`02_silver`.`taxi_zones`
             GROUP BY LocationID
             HAVING COUNT(*) > 1
         )) AS duplicate_location_id,

        -- LocationID validation
        (SELECT COUNT(DISTINCT LocationID)
         FROM `ftw-week-08`.`02_silver`.`taxi_zones`) AS distinct_location_ids,

        (SELECT MIN(LocationID)
         FROM `ftw-week-08`.`02_silver`.`taxi_zones`) AS min_location_id,

        (SELECT MAX(LocationID)
         FROM `ftw-week-08`.`02_silver`.`taxi_zones`) AS max_location_id,

        -- Referential integrity: Silver Green Taxi → Silver Taxi Zones
        (SELECT COUNT(*)
         FROM `ftw-week-08`.`02_silver`.`green_taxi` g
         LEFT JOIN `ftw-week-08`.`02_silver`.`taxi_zones` z
             ON g.PULocationID = z.LocationID
         WHERE g.PULocationID IS NOT NULL
           AND z.LocationID IS NULL) AS orphan_pickup,

        (SELECT COUNT(*)
         FROM `ftw-week-08`.`02_silver`.`green_taxi` g
         LEFT JOIN `ftw-week-08`.`02_silver`.`taxi_zones` z
             ON g.DOLocationID = z.LocationID
         WHERE g.DOLocationID IS NOT NULL
           AND z.LocationID IS NULL) AS orphan_dropoff
)

SELECT
    check_name,
    expected,
    actual,
    CASE
        WHEN expected = actual THEN 'PASS'
        ELSE 'FAIL'
    END AS status
FROM (
    SELECT 'Source row count' AS check_name,
           265 AS expected,
           source_count AS actual
    FROM validation_results

    UNION ALL

    SELECT 'Silver row count',
           265,
           silver_count
    FROM validation_results

    UNION ALL

    SELECT 'NULL LocationID',
           0,
           null_location_id
    FROM validation_results

    UNION ALL

    SELECT 'NULL Borough',
           0,
           null_borough
    FROM validation_results

    UNION ALL

    SELECT 'NULL Zone',
           0,
           null_zone
    FROM validation_results

    UNION ALL

    SELECT 'NULL service_zone',
           0,
           null_service_zone
    FROM validation_results

    UNION ALL

    SELECT 'NULL source_system',
           0,
           null_source_system
    FROM validation_results

    UNION ALL

    SELECT 'NULL source_file',
           0,
           null_source_file
    FROM validation_results

    UNION ALL

    SELECT 'NULL batch_id',
           0,
           null_batch_id
    FROM validation_results

    UNION ALL

    SELECT 'NULL ingested_at',
           0,
           null_ingested_at
    FROM validation_results

    UNION ALL

    SELECT 'Invalid service_zone',
           0,
           invalid_service_zone
    FROM validation_results

    UNION ALL

    SELECT 'Duplicate LocationID',
           0,
           duplicate_location_id
    FROM validation_results

    UNION ALL

    SELECT 'Distinct LocationID',
           265,
           distinct_location_ids
    FROM validation_results

    UNION ALL

    SELECT 'Minimum LocationID',
           1,
           min_location_id
    FROM validation_results

    UNION ALL

    SELECT 'Maximum LocationID',
           265,
           max_location_id
    FROM validation_results

    UNION ALL

    SELECT 'Orphan pickup IDs',
           0,
           orphan_pickup
    FROM validation_results

    UNION ALL

    SELECT 'Orphan dropoff IDs',
           0,
           orphan_dropoff
    FROM validation_results
)
ORDER BY
    CASE check_name
        WHEN 'Source row count' THEN 1
        WHEN 'Silver row count' THEN 2
        WHEN 'NULL LocationID' THEN 3
        WHEN 'NULL Borough' THEN 4
        WHEN 'NULL Zone' THEN 5
        WHEN 'NULL service_zone' THEN 6
        WHEN 'NULL source_system' THEN 7
        WHEN 'NULL source_file' THEN 8
        WHEN 'NULL batch_id' THEN 9
        WHEN 'NULL ingested_at' THEN 10
        WHEN 'Invalid service_zone' THEN 11
        WHEN 'Duplicate LocationID' THEN 12
        WHEN 'Distinct LocationID' THEN 13
        WHEN 'Minimum LocationID' THEN 14
        WHEN 'Maximum LocationID' THEN 15
        WHEN 'Orphan pickup IDs' THEN 16
        WHEN 'Orphan dropoff IDs' THEN 17
    END;

In [0]:
SELECT DISTINCT service_zone
FROM `ftw-week-08`.`02_silver`.taxi_zones